# RetailPulse — Data Cleaning & Feature Engineering

## 1. Imports

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# add project root to path so we can import from src/
sys.path.append(os.path.abspath('..'))
from src.feature_engineering import load_sales, clean_sales, build_rfm, build_rolling_stats

sns.set_theme(style='whitegrid')
print('imports done')

## 2. Load Raw Datasets

Checking shape before any cleaning.

In [ ]:
df_sales_raw = load_sales('../data/raw/online_retail_II.csv')
df_churn     = pd.read_csv('../data/raw/online_retail_customer_churn.csv')
df_inventory = pd.read_csv('../data/raw/retail_store_inventory.csv')

print('Shape BEFORE cleaning:')
print(f'  Sales     : {df_sales_raw.shape}')
print(f'  Churn     : {df_churn.shape}')
print(f'  Inventory : {df_inventory.shape}')

## 3. Clean Sales Dataset

Using `clean_sales()` from `src/feature_engineering.py`:
- Removes cancellations (negative quantity)
- Removes zero-price rows
- Drops missing Customer IDs
- Removes duplicates
- Adds Revenue column

In [ ]:
df_sales_clean = clean_sales(df_sales_raw)

print(f'\nShape BEFORE : {df_sales_raw.shape}')
print(f'Shape AFTER  : {df_sales_clean.shape}')
print(f'Rows removed : {df_sales_raw.shape[0] - df_sales_clean.shape[0]:,}')

In [ ]:
display(df_sales_clean.head(5))
print('\nmissing values:', df_sales_clean.isnull().sum().sum())

Churn and Inventory datasets are already clean (confirmed in EDA), so no additional cleaning needed.

In [ ]:
print('Churn missing values    :', df_churn.isnull().sum().sum())
print('Inventory missing values:', df_inventory.isnull().sum().sum())

## 4. RFM Feature Engineering

Recency, Frequency, Monetary scores (1-5) + customer segment label.

In [ ]:
rfm = build_rfm(df_sales_clean)

print(f'\nRFM table shape: {rfm.shape}')
display(rfm.head(8))

In [ ]:
seg_counts = rfm['Segment'].value_counts()

plt.figure(figsize=(10, 5))
colors = ['#2ecc71', '#3498db', '#9b59b6', '#e67e22', '#e74c3c']
seg_counts.plot(kind='bar', color=colors[:len(seg_counts)], edgecolor='white')
plt.title('Customer Segment Distribution (RFM)', fontsize=14, fontweight='bold')
plt.xlabel('Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=30)
for i, v in enumerate(seg_counts):
    plt.text(i, v + 10, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/rfm_segments.png', dpi=150)
plt.show()

## 5. Rolling Statistics

7-day and 30-day rolling mean on daily revenue. Will be used for demand forecasting.

In [ ]:
daily = build_rolling_stats(df_sales_clean, windows=[7, 30])

print(f'Daily stats shape: {daily.shape}')
display(daily.tail(5))

In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(daily['Date'], daily['Revenue'], color='lightsteelblue', linewidth=1, alpha=0.6, label='Daily Revenue')
plt.plot(daily['Date'], daily['rolling_7d_mean'],  color='steelblue', linewidth=2, label='7-day avg')
plt.plot(daily['Date'], daily['rolling_30d_mean'], color='coral',     linewidth=2, label='30-day avg')

plt.title('Daily Revenue with Rolling Averages', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Revenue (GBP)')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/rolling_statistics.png', dpi=150)
plt.show()

## 6. Load & Preview Processed Outputs

In [ ]:
rfm_out   = pd.read_csv('../data/processed/rfm_scores.csv')
daily_out = pd.read_csv('../data/processed/daily_revenue_rolling.csv')
clean_out = pd.read_csv('../data/processed/retail_clean.csv', low_memory=False)

print(f'rfm_scores.csv            : {rfm_out.shape}')
print(f'daily_revenue_rolling.csv : {daily_out.shape}')
print(f'retail_clean.csv          : {clean_out.shape}')

In [ ]:
print('--- rfm_scores sample ---')
display(rfm_out.head(5))

In [ ]:
print('--- daily_revenue_rolling sample ---')
display(daily_out.head(5))

In [ ]:
print('--- retail_clean sample ---')
display(clean_out.head(5))

## Done!

- Sales cleaned: 1,067,371 -> 779,425 rows
- RFM built for 5,878 customers with 5 segments
- Rolling stats for 604 days
- All outputs in data/processed/

Next -> Day 3: Customer Segmentation (K-Means + DBSCAN)